# Scaling Physical AI & Robotics Systems with Ray

## TLDR

This is the overview for a six-notebook course on building an **end-to-end
physical-AI workflow** with **Ray on Anyscale**. Across the series you stream
robotics data, fine-tune a Vision-Language-Action (VLA) policy, serve it and
evaluate it in simulation, close the training loop with sim data, pre-train a
**world model** at scale, and **distill** a large model into a small backbone
fit for a robot. The models are the workload. The lesson is the infrastructure —
**Ray Data, Ray Train, Ray Serve, and Ray Core** — that makes all of it
run on one cluster.

> **Scale up to learn. Scale down to deploy.**

## Why the infrastructure is the hard part

Talk to robotics teams and you hear the same thing: the models aren't the
bottleneck, but rather the **infrastructure around them** is. And that infrastructure burden is really a **scaling** problem, as most teams have exceeded the model and data size to justify single instance workflows (i.e. less than or equal to 8 GPUs). A handful of core workloads
recur at most robotics orgs, and this course walks one example of each:

| Core workload | Where this course shows it |
|---|---|
| Sensor-log & video processing | 01 |
| Dataset curation & auto-labeling | 01, 03 |
| Distributed training (perception, VLA, world models) | 02, 04, 05 |
| Parallel simulation (RL, sim-eval, data synthesis) | 03 |
| Post-deployment (fleet telemetry pipelines) | 03, 05 |

Run any of these at real scale and the **same three pains** surface:

- **DevEx & runtime that slow iteration speed**: bespoke data loaders, glue
  scripts, and long waits on data copies between every experiment.
- **Wrangling infra instead of scaling the solution**: DDP boilerplate, device
  placement, process isolation, fragile checkpointing and retries.
- **Underutilized, spiky, high-touch compute**: clusters sit idle, then thrash with subopimal performance;
  GPUs are reserved, hand-placed, and babysat.

Take **parallel simulation**: naively implemented, it has poor DevEx (custom orchestration per run),
it's pure infra-wrangling (GPU placement, subprocess isolation, HTTP plumbing to
the policy), *and* it's spiky (idle between eval rounds, then a burst across many
GPUs). Every other row is the same story in a different shape, which is the real
reason this is hard. Heterogeneous workloads (CPU video decode, multi-GPU
training, GPU simulation, HTTP serving) that must **share one cluster and pass
state between each other**, each carrying all three pains at once.

**Ray on Anyscale attacks all three across every workload:**

- **Iteration speed.** Ray Data *streams* logs and video straight from cloud
  storage (no local copies, no bespoke loaders), and the same `map_batches`
  curation logic also auto-labels and filters sim rollouts back into training.
- **Scaling, not infra-wrangling.** Ray Train and Ray Serve hide the distributed
  boilerplate — `prepare_model` for DDP, `ScalingConfig(num_workers=N)` from 4
  GPUs to 400, fault-tolerant checkpointing for free — behind one API reused for
  perception, VLA, world-model, and distillation jobs. Change the config, not the
  code.
- **Elastic, well-utilized compute.** Ray schedules these heterogeneous tasks on
  one cluster and **releases GPUs between phases** (here, training uses all 4
  GPUs; sim-eval hands 3 back), while Anyscale autoscales instead of leaving a cluster reserved and idle.

Most teams have the models. What they lack is the above. The rest of the course is the
story end to end where each section is one of these workloads run on the Ray
primitives that remove all three pains.

## The lifecycle this course builds

```
   ┌─────────┐   ┌──────────────┐   ┌────────────┐   ┌─────────────┐
   │  DATA   │──▶│  FINE-TUNE   │──▶│   SERVE    │──▶│   SIMULATE  │
   │ Ray Data│   │  a VLA       │   │ the policy │   │  & evaluate │
   │ stream  │   │  Ray Train   │   │ Ray Serve  │   │  Ray core   │
   └─────────┘   └──────────────┘   └────────────┘   └──────┬──────┘
        ▲                                                   │
        │                  CLOSE THE LOOP                   │
        └───────────  filter by reward, union  ◀────────────┘
                              │
            ┌─────────────────┴───────────────────┐
            ▼                                      ▼
   ┌──────────────────┐                  ┌────────────────────┐
   │  WORLD MODEL     │                  │   DISTILL FOR EDGE │
   │  pre-train at    │  ───────────▶    │  big teacher →     │
   │  scale (V-JEPA)  │   scale down     │  small student     │
   │  Ray Train       │                  │  Ray Train         │
   └──────────────────┘                  └────────────────────┘
```

The arc runs from **using a simulator as a data factory** to **learning the
simulator itself** (a world model), and finally to **shrinking models down**
so they fit on a robot. The same Ray Train code carries you from a 3.4B model on
many GPUs to a mobile network that runs on the edge.

## What you will learn

By the end of the series you will be able to:

- **Stream** large multi-camera robotics datasets with **Ray Data** — no local
  copy on any node — and preprocess them on a resource pool that scales independently
  of your GPUs.
- **Fine-tune and pre-train** large robotics models with **Ray Train** — DDP,
  mixed precision, fault-tolerant checkpointing — scaling from a few GPUs to
  hundreds by changing one config value.
- **Serve** a policy as an HTTP service with **Ray Serve**, the same primitive
  used to serve LLMs in production.
- **Fan out** parallel simulation rollouts as **Ray remote tasks**, each on its
  own GPU, fully fault-isolated.
- **Close the loop**: filter sim trajectories by reward and mix them back
  into the training stream.
- **Distill** a large teacher model into a small, deployable student backbone for real-time inference.

## The through-line: one Ray surface, taught once, reused everywhere

The whole course rests on a small, repeating set of Ray APIs. Learn them in
01–02 and you will recognize them unchanged in every later notebook:

| Notebook | Ray primitive(s) | What it does | Outline bullet |
|----------|------------------|--------------|----------------|
| 01 Data pipelines | `ray.data` / `read_lerobot` / `map_batches` | Stream + preprocess LeRobot v3 video | Robotics data prep |
| 02 VLA fine-tuning | `TorchTrainer`, `prepare_model`, `train.report`, `ScalingConfig` | DDP fine-tune PI0.5 | VLA fine-tuning |
| 03 Serve + sim eval | `@serve.deployment`, `serve.run`, `@ray.remote(num_gpus=1)` | Policy server + Isaac Lab fan-out + close the loop | Distributed sim & eval; scalable inference |
| 04 World model | `read_lerobot`, `TorchTrainer` (×2 phases) | Pre-train V-JEPA + online adaptation | World-model pre-training at scale; VLA pre-training |
| 05 Distillation | `ray.data`, `TorchTrainer` | Teacher→student for edge deploy | Scalable inference (edge) |

The same `TorchTrainer` + `prepare_model` + `train.report` + `FailureConfig` +
`ScalingConfig` drives fine-tuning (02), world-model pre-training (04), and
distillation (05). The same `lerobot_datasource` feeds 01–04. **Change the
config, not the code.**

## How this scales on Anyscale

Every notebook runs at deliberately small scale so it finishes in
minutes on a 4× L4 cluster. The same code path scales on Anyscale by adjusting
configuration, not rewriting logic:

| Lever | This course | Production |
|-------|-------------|------------|
| Dataset | streamed subset | full corpus (10 TB+), same `hf://` / S3 path |
| Train workers | 4 × L4 | 8–64+ × A100/H100 — change `ScalingConfig(num_workers=N)` |
| Train steps | ~50–200  | full epochs |
| Sim workers | 2 | as many GPUs as you have — change `SIM_WORKERS` |
| Serve replicas | 1 | autoscaled behind a load balancer |

Anyscale manages cluster scaling, GPU scheduling, and shared storage. You just need to change
a few numbers.

## Course map — how to navigate

Read the notebooks in order. Each builds on the last and cross-references it.
All ship **pre-run** (outputs committed), so you can read the full story without
a cluster, then re-run any notebook yourself.

| # | Notebook | Focus |
|---|----------|-------|
| 00 | this notebook | The lifecycle and the through-line |
| 01 | `01_robotics_data_pipelines.ipynb` | Stream + preprocess robotics video (Ray Data) |
| 02 | `02_vla_finetuning.ipynb` | Distributed DDP fine-tune of PI0.5 (Ray Train) |
| 03 | `03_serving_and_sim_eval.ipynb` | Serve + Isaac Lab fan-out + close the loop |
| 04 | `04_world_model_pretraining.ipynb` | V-JEPA world model pre-training at scale |
| 05 | `05_distillation_for_edge.ipynb` | Distill a teacher into an edge-ready student |

**Prerequisites:** a 4× L4 Anyscale cluster on the course image, and (for 02/03)
`export HF_TOKEN=...` with access to the gated `google/paligemma-3b-pt-224`. See
`README.md` for the full cluster image and tested versions.

**A note on scope:** both the LIBERO data and Isaac Lab's Franka task use a
Franka Panda, so the dimensions match — but PI0.5 hasn't seen this exact setup
(Isaac Lab's control convention, scene, and camera views), so expect exploratory
motion, not task success. We validate the orchestration loop, not manipulation.

## Cell 1: Confirm your cluster

**What you do**: connect to the Ray cluster and print its resources.

**What to check**: you should see **4 GPUs** and a pool of CPUs. On Anyscale,
`ray.init(address="auto")` attaches to the managed cluster that is already
running.

**Why it matters**: this single line is all the infrastructure setup the rest
of the course needs. Everything downstream (Ray Data preprocessing, Ray Train
DDP, Ray Serve, sim tasks) draws from these resources.

In [1]:
import ray

try:
    ray.init(address="auto", ignore_reinit_error=True)
except ConnectionError:
    ray.init(ignore_reinit_error=True)

res = ray.cluster_resources()
print(f"GPUs:   {int(res.get('GPU', 0))}")
print(f"CPUs:   {int(res.get('CPU', 0))}")
print(f"Memory: {res.get('memory', 0) / 1e9:.0f} GiB")

2026-07-30 10:38:18,256	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 10.0.64.141:6379...
2026-07-30 10:38:18,281	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at https://session-fn84hi73bzxr1v1qbpgcmzd24c.i.anyscaleuserdata.com 
2026-07-30 10:38:18,288	INFO packaging.py:496 -- Pushing file package 'gcs://_ray_pkg_505ac2507c54d8c02ec17700c850e5f458b417f9.zip' (0.59MiB) to Ray cluster...
2026-07-30 10:38:18,290	INFO packaging.py:509 -- Successfully pushed file package 'gcs://_ray_pkg_505ac2507c54d8c02ec17700c850e5f458b417f9.zip'.


GPUs:   4
CPUs:   48
Memory: 241 GiB


(autoscaler +1m30s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


## Where to go next

Continue to **`01_robotics_data_pipelines.ipynb`**, where Ray Data streams the
LIBERO dataset straight from HuggingFace. That's hundreds of thousands of Franka
manipulation frames without ever landing a copy on disk.